In [ ]:
import os
import re
from collections import Counter

import pandas as pd

import cleaning_pipeline as cp
from data_loader import load_imdb_data

# Load the raw data
df = load_imdb_data()

# Recreate the output of Member 1's Steps 2-4 with the same functions (from cleaning_pipeline.py)
df_processed = df.copy()
df_processed["normalized_review"] = df_processed["review"].apply(cp.normalize_unicode)   # Step 2
df_processed["cleaned_review"] = df_processed["normalized_review"].apply(cp.clean_text)   # Step 3
df_processed["regex_review"] = df_processed["cleaned_review"].apply(cp.regex_process)     # Step 4

rating_pattern = r"\b\d{1,2}/10\b"   # rating pattern from Step 4

print("Shape:", df_processed.shape)
print("Columns:", df_processed.columns.tolist())

## Step 5: Lowercase Normalization

### 5.1 A small example

In [44]:
words = ["Great", "GREAT", "great", "GrEaT"]

print("Before lowercasing:", words, "-> unique words:", len(set(words)))

lower_words = [word.lower() for word in words]
print("After lowercasing: ", lower_words, "-> unique words:", len(set(lower_words)))

Before lowercasing: ['Great', 'GREAT', 'great', 'GrEaT'] -> unique words: 4
After lowercasing:  ['great', 'great', 'great', 'great'] -> unique words: 1


### 5.2 Apply lowercasing to the reviews

In [45]:
df_processed["lowercase_review"] = df_processed["regex_review"].str.lower()

df_processed[["regex_review", "lowercase_review"]].head()

,regex_review,lowercase_review
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
1,A wonderful little production. The filming tec...,a wonderful little production. the filming tec...
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...","petter mattei's ""love in the time of money"" is..."


### 5.3 Before-and-after examples

In [46]:
def show_lowercase_change(index, n_chars=250):
    print("Review index:", index)
    print("BEFORE:", df_processed.loc[index, "regex_review"][:n_chars])
    print("AFTER: ", df_processed.loc[index, "lowercase_review"][:n_chars])
    print("-" * 80)


print("Example 1: first review")
show_lowercase_change(0)

print("Example 2: a review with a word in CAPITALS")
capital_pattern = r"\b[A-Z]{4,}\b"
capital_index = df_processed["regex_review"].apply(lambda text: re.search(capital_pattern, text) is not None).idxmax()
match = re.search(capital_pattern, df_processed.loc[capital_index, "regex_review"])
start = max(0, match.start() - 80)
end = match.end() + 80
print("Review index:", capital_index)
print("CAPITAL word found:", match.group())
print("BEFORE:", df_processed.loc[capital_index, "regex_review"][start:end])
print("AFTER: ", df_processed.loc[capital_index, "lowercase_review"][start:end])
print("-" * 80)

Example 1: first review
Review index: 0
BEFORE: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me. The first thing that struck me about Oz was its brutality and unflinching scenes of violence, w
AFTER:  one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me. the first thing that struck me about oz was its brutality and unflinching scenes of violence, w
--------------------------------------------------------------------------------
Example 2: a review with a word in CAPITALS
Review index: 3
CAPITAL word found: BOOGEYMAN
BEFORE: have Jake with his closet which totally ruins all the film! I expected to see a BOOGEYMAN similar movie, and instead i watched a drama with some meaningless thriller spo
AFTER:  have jake with his closet which totally ruins all the film! i expected to see a booge

### 5.4 The difference in the whole dataset

In [47]:
def count_word(column, word):
    pattern = r"\b" + word + r"\b"
    return df_processed[column].apply(lambda text: len(re.findall(pattern, text))).sum()


print("BEFORE lowercasing (regex_review):")
for word in ["great", "Great", "GREAT"]:
    print(f"  '{word}': {count_word('regex_review', word)} times")

print("\nAFTER lowercasing (lowercase_review):")
for word in ["great", "Great", "GREAT"]:
    print(f"  '{word}': {count_word('lowercase_review', word)} times")

BEFORE lowercasing (regex_review):
  'great': 16797 times
  'Great': 1154 times
  'GREAT': 194 times

AFTER lowercasing (lowercase_review):
  'great': 18145 times
  'Great': 0 times
  'GREAT': 0 times


### 5.5 Check the lowercase results

In [48]:
lowercase_text = df_processed["lowercase_review"]

print("Reviews still containing capital letters:", (lowercase_text != lowercase_text.str.lower()).sum())
print("Reviews whose length changed:            ", (lowercase_text.str.len() != df_processed["regex_review"].str.len()).sum())
print("Empty reviews:                           ", (lowercase_text.str.len() == 0).sum())

print("\nOriginal review column unchanged:", df["review"].equals(df_processed["review"]))
print("df_processed columns:", df_processed.columns.tolist())

Reviews still containing capital letters: 0
Reviews whose length changed:             0
Empty reviews:                            0

Original review column unchanged: True
df_processed columns: ['review', 'sentiment', 'normalized_review', 'cleaned_review', 'regex_review', 'lowercase_review']


## Step 6: Tokenization

### 6.1 Import NLTK

In [49]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("NLTK version:", nltk.__version__)

NLTK version: 3.10.0


### 6.2 A small example: `split()` vs `word_tokenize()`

In [50]:
example_text = "i don't think it's a great movie... i'd give it 3/10! the old-time feel was nice, though."

print("Text:", example_text)
print()
print("split()        :", example_text.split())
print()
print("word_tokenize():", word_tokenize(example_text))

Text: i don't think it's a great movie... i'd give it 3/10! the old-time feel was nice, though.

split()        : ['i', "don't", 'think', "it's", 'a', 'great', 'movie...', "i'd", 'give', 'it', '3/10!', 'the', 'old-time', 'feel', 'was', 'nice,', 'though.']

word_tokenize(): ['i', 'do', "n't", 'think', 'it', "'s", 'a', 'great', 'movie', '...', 'i', "'d", 'give', 'it', '3/10', '!', 'the', 'old-time', 'feel', 'was', 'nice', ',', 'though', '.']


### 6.3 Method 1 vs Method 2 on real IMDb sentences: regex tokenizer vs NLTK

In [51]:
regex_token_pattern = r"\w+|[^\w\s]"


def regex_tokenize(text):
    return re.findall(regex_token_pattern, text)


def find_real_sentence(pattern, window=45):
    for text in df_processed["lowercase_review"]:
        match = re.search(pattern, text)
        if match:
            return text[max(0, match.start() - window): match.end() + window]
    return None


real_sentences = [
    find_real_sentence(r"\bdon't\b"),      # contraction
    find_real_sentence(rating_pattern),     # rating like 7/10 (pattern from Step 4)
    find_real_sentence(r"\b\w+-\w+\b"),     # hyphenated word
    find_real_sentence(r'"\w'),             # quotation marks
]

for number, sentence in enumerate(real_sentences, start=1):
    if sentence is None:
        continue
    regex_tokens = regex_tokenize(sentence)
    nltk_tokens = word_tokenize(sentence)
    print(f"Sentence {number}: {sentence}")
    print(f"  Regex tokenizer ({len(regex_tokens)} tokens): {regex_tokens}")
    print(f"  NLTK tokenizer  ({len(nltk_tokens)} tokens): {nltk_tokens}")
    print("-" * 80)

Sentence 1: w carver in a first person perspective so we don't really know what he looked like when he was 
  Regex tokenizer (21 tokens): ['w', 'carver', 'in', 'a', 'first', 'person', 'perspective', 'so', 'we', 'don', "'", 't', 'really', 'know', 'what', 'he', 'looked', 'like', 'when', 'he', 'was']
  NLTK tokenizer  (20 tokens): ['w', 'carver', 'in', 'a', 'first', 'person', 'perspective', 'so', 'we', 'do', "n't", 'really', 'know', 'what', 'he', 'looked', 'like', 'when', 'he', 'was']
--------------------------------------------------------------------------------
Sentence 2: us at the same time, i'm giving this movie a 3/10, an improvement from the 0.5/10 i was origin
  Regex tokenizer (29 tokens): ['us', 'at', 'the', 'same', 'time', ',', 'i', "'", 'm', 'giving', 'this', 'movie', 'a', '3', '/', '10', ',', 'an', 'improvement', 'from', 'the', '0', '.', '5', '/', '10', 'i', 'was', 'origin']
  NLTK tokenizer  (22 tokens): ['us', 'at', 'the', 'same', 'time', ',', 'i', "'m", 'giving', 'this',

### 6.4 Apply tokenization to the reviews

In [52]:
def tokenize_review(text):
    tokens = word_tokenize(text)
    # "fan- always" gives the token "fan-", so remove a hyphen stuck at the start or end of a token
    return [token.strip("-") if token.strip("-") else token for token in tokens]


df_processed["tokens"] = df_processed["lowercase_review"].apply(tokenize_review)

df_processed[["lowercase_review", "tokens"]].head()

,lowercase_review,tokens
0,one of the other reviewers has mentioned that ...,"[one, of, the, other, reviewers, has, mentione..."
1,a wonderful little production. the filming tec...,"[a, wonderful, little, production, ., the, fil..."
2,i thought this was a wonderful way to spend ti...,"[i, thought, this, was, a, wonderful, way, to,..."
3,basically there's a family where a little boy ...,"[basically, there, 's, a, family, where, a, li..."
4,"petter mattei's ""love in the time of money"" is...","[petter, mattei, 's, ``, love, in, the, time, ..."


### 6.5 Examples: original review and its tokens

In [53]:
def show_tokens(index, n_chars=300, n_tokens=40):
    tokens = df_processed.loc[index, "tokens"]
    print("Review index:", index)
    print("ORIGINAL REVIEW: ", df_processed.loc[index, "review"][:n_chars])
    print()
    print("LOWERCASE REVIEW:", df_processed.loc[index, "lowercase_review"][:n_chars])
    print()
    print("TOKENS (first", n_tokens, "):", tokens[:n_tokens])
    print()
    print("Total tokens in this review:", len(tokens))
    print("-" * 80)


# Example 1 and 2: the first two reviews
show_tokens(0)
show_tokens(1)

# Example 3: the first review that contains a rating like 7/10
rating_index = df_processed["lowercase_review"].apply(lambda text: re.search(rating_pattern, text) is not None).idxmax()
show_tokens(rating_index)

# The rating is usually later in the review, so also show the part of the text around it
text = df_processed.loc[rating_index, "lowercase_review"]
match = re.search(rating_pattern, text)
part = text[max(0, match.start() - 60): match.end() + 60]
print("Part of review", rating_index, "around the rating", match.group())
print("TEXT:  ", part)
print("TOKENS:", tokenize_review(part))

Review index: 0
ORIGINAL REVIEW:  One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru

LOWERCASE REVIEW: one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me. the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this

TOKENS (first 40 ): ['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', 'you', "'ll", 'be', 'hooked', '.', 'they', 'are', 'right', ',', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me', '.', 'the', 'first', 'thing', 'that', 'struck', 'me', 'about', 'oz']

Tot

### 6.6 How contractions and ratings are tokenized in the dataset

In [54]:
contraction_index = df_processed["lowercase_review"].apply(lambda text: "n't" in text).idxmax()
text = df_processed.loc[contraction_index, "lowercase_review"]
position = text.index("n't")
part = text[max(0, position - 60): position + 60]

print("Review index:", contraction_index)
print("TEXT:  ", part)
print("TOKENS:", tokenize_review(part))

Review index: 0
TEXT:   show is due to the fact that it goes where other shows wouldn't dare. forget pretty pictures painted for mainstream audi
TOKENS: ['show', 'is', 'due', 'to', 'the', 'fact', 'that', 'it', 'goes', 'where', 'other', 'shows', 'would', "n't", 'dare', '.', 'forget', 'pretty', 'pictures', 'painted', 'for', 'mainstream', 'audi']


### 6.7 Tokenization challenges in our IMDb data

In [55]:
challenge_examples = [
    "i didn't like it and i wouldn't watch it again",
    "i am an anime fan- always has been",
    "i'd give it 3/10, maybe 0.5/10",
    'the "director" spent the whole time on a trip',
]

for text in challenge_examples:
    print("Text:            ", text)
    print("word_tokenize(): ", word_tokenize(text))
    print("tokenize_review():", tokenize_review(text))
    print()

Text:             i didn't like it and i wouldn't watch it again
word_tokenize():  ['i', 'did', "n't", 'like', 'it', 'and', 'i', 'would', "n't", 'watch', 'it', 'again']
tokenize_review(): ['i', 'did', "n't", 'like', 'it', 'and', 'i', 'would', "n't", 'watch', 'it', 'again']

Text:             i am an anime fan- always has been
word_tokenize():  ['i', 'am', 'an', 'anime', 'fan-', 'always', 'has', 'been']
tokenize_review(): ['i', 'am', 'an', 'anime', 'fan', 'always', 'has', 'been']

Text:             i'd give it 3/10, maybe 0.5/10
word_tokenize():  ['i', "'d", 'give', 'it', '3/10', ',', 'maybe', '0.5/10']
tokenize_review(): ['i', "'d", 'give', 'it', '3/10', ',', 'maybe', '0.5/10']

Text:             the "director" spent the whole time on a trip
word_tokenize():  ['the', '``', 'director', "''", 'spent', 'the', 'whole', 'time', 'on', 'a', 'trip']
tokenize_review(): ['the', '``', 'director', "''", 'spent', 'the', 'whole', 'time', 'on', 'a', 'trip']



### 6.8 Token statistics

In [56]:
token_counts = df_processed["tokens"].apply(len)

print("Tokens per review:")
print(token_counts.describe().round(1))

unique_tokens = set()
for tokens in df_processed["tokens"]:
    unique_tokens.update(tokens)

print("\nTotal tokens in all reviews: ", token_counts.sum())
print("Unique tokens (vocabulary):  ", len(unique_tokens))

Tokens per review:
count    50000.0
mean       263.8
std        195.5
min          8.0
25%        144.0
50%        198.0
75%        321.0
max       2738.0
Name: tokens, dtype: float64

Total tokens in all reviews:  13191275
Unique tokens (vocabulary):   152941


### 6.9 Check the tokenization results

In [57]:
print("All values are lists:          ", df_processed["tokens"].apply(lambda tokens: isinstance(tokens, list)).all())
print("Reviews with no tokens:         ", (token_counts == 0).sum())
print("Tokens containing a space:      ", sum(1 for token in unique_tokens if " " in token))

print("\nOriginal review column unchanged:", df["review"].equals(df_processed["review"]))
print("df_processed columns:", df_processed.columns.tolist())

All values are lists:           True
Reviews with no tokens:          0
Tokens containing a space:       0

Original review column unchanged: True
df_processed columns: ['review', 'sentiment', 'normalized_review', 'cleaned_review', 'regex_review', 'lowercase_review', 'tokens']


## Step 7: Stop Words Removal

### 7.1 Download and load NLTK's English stopword list

In [58]:
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

english_stopwords = set(stopwords.words("english"))

print("Number of English stopwords in NLTK:", len(english_stopwords))
print("\nSome stopwords:", sorted(english_stopwords)[:40])

Number of English stopwords in NLTK: 198

Some stopwords: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don']


### 7.2 Keep the negation words

In [59]:
negation_words = {
    "not", "no", "nor", "never", "none", "nothing", "nobody", "nowhere", "neither", "cannot", "n't",
    "ain", "aren", "couldn", "didn", "doesn", "don", "hadn", "hasn", "haven", "isn",
    "mightn", "mustn", "needn", "shan", "shouldn", "wasn", "weren", "won", "wouldn",
}

stop_words_to_remove = {
    word for word in english_stopwords
    if word not in negation_words and not word.endswith("n't")
}

kept_from_nltk_list = sorted(english_stopwords - stop_words_to_remove)

print("Stopwords in NLTK list:     ", len(english_stopwords))
print("Stopwords we will remove:   ", len(stop_words_to_remove))
print("Negation words we KEEP:     ", len(kept_from_nltk_list))
print("\nNegation words kept from the NLTK list:")
print(kept_from_nltk_list)

Stopwords in NLTK list:      198
Stopwords we will remove:    158
Negation words we KEEP:      40

Negation words kept from the NLTK list:
['ain', 'aren', "aren't", 'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'don', "don't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't", 'isn', "isn't", 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't", 'no', 'nor', 'not', 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"]


### 7.3 Small example: why keeping negations matters

In [60]:
example_reviews = [
    ["this", "movie", "is", "not", "good", "."],
    ["i", "do", "not", "like", "the", "acting", "and", "no", "one", "should", "watch", "it", "."],
]

for tokens in example_reviews:
    with_full_list = [token for token in tokens if token not in english_stopwords]
    with_our_list = [token for token in tokens if token not in stop_words_to_remove]
    print("Tokens:                    ", tokens)
    print("Full NLTK list (WRONG):    ", with_full_list)
    print("Our list, negations kept:  ", with_our_list)
    print()

Tokens:                     ['this', 'movie', 'is', 'not', 'good', '.']
Full NLTK list (WRONG):     ['movie', 'good', '.']
Our list, negations kept:   ['movie', 'not', 'good', '.']

Tokens:                     ['i', 'do', 'not', 'like', 'the', 'acting', 'and', 'no', 'one', 'should', 'watch', 'it', '.']
Full NLTK list (WRONG):     ['like', 'acting', 'one', 'watch', '.']
Our list, negations kept:   ['not', 'like', 'acting', 'no', 'one', 'watch', '.']



### 7.4 Remove stopwords from all reviews

In [61]:
def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words_to_remove]


df_processed["tokens_no_stopwords"] = df_processed["tokens"].apply(remove_stopwords)

df_processed[["tokens", "tokens_no_stopwords"]].head()

,tokens,tokens_no_stopwords
0,"[one, of, the, other, reviewers, has, mentione...","[one, reviewers, mentioned, watching, 1, oz, e..."
1,"[a, wonderful, little, production, ., the, fil...","[wonderful, little, production, ., filming, te..."
2,"[i, thought, this, was, a, wonderful, way, to,...","[thought, wonderful, way, spend, time, hot, su..."
3,"[basically, there, 's, a, family, where, a, li...","[basically, 's, family, little, boy, (, jake, ..."
4,"[petter, mattei, 's, ``, love, in, the, time, ...","[petter, mattei, 's, ``, love, time, money, ''..."


### 7.5 Before-and-after examples

In [62]:
def show_stopword_removal(index, n_tokens=30):
    before = df_processed.loc[index, "tokens"][:n_tokens]
    removed = [token for token in before if token in stop_words_to_remove]
    remaining = [token for token in before if token not in stop_words_to_remove]
    print("Review index:", index)
    print("BEFORE (first", n_tokens, "tokens):", before)
    print()
    print("REMOVED STOPWORDS:", removed)
    print()
    print("REMAINING TOKENS: ", remaining)
    print()
    print("Tokens in whole review:", len(df_processed.loc[index, "tokens"]),
          "->", len(df_processed.loc[index, "tokens_no_stopwords"]))
    print("-" * 80)


# Example 1 and 2: the first two reviews
show_stopword_removal(0)
show_stopword_removal(1)

Review index: 0
BEFORE (first 30 tokens): ['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', 'you', "'ll", 'be', 'hooked', '.', 'they', 'are', 'right', ',', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with']

REMOVED STOPWORDS: ['of', 'the', 'other', 'has', 'that', 'after', 'just', 'you', 'be', 'they', 'are', 'as', 'this', 'is', 'what', 'with']

REMAINING TOKENS:  ['one', 'reviewers', 'mentioned', 'watching', '1', 'oz', 'episode', "'ll", 'hooked', '.', 'right', ',', 'exactly', 'happened']

Tokens in whole review: 359 -> 220
--------------------------------------------------------------------------------
Review index: 1
BEFORE (first 30 tokens): ['a', 'wonderful', 'little', 'production', '.', 'the', 'filming', 'technique', 'is', 'very', 'unassuming', 'very', 'old-time-bbc', 'fashion', 'and', 'gives', 'a', 'comforting', ',', 'and', 'sometimes', 'discomforting', ',', 'sense', 'of', 'realism', 'to', 'the', 'enti

In [63]:
not_index = df_processed["tokens"].apply(lambda tokens: "not" in tokens).idxmax()
tokens = df_processed.loc[not_index, "tokens"]
position = tokens.index("not")
part = tokens[max(0, position - 10): position + 10]

print("Review index:", not_index)
print("BEFORE:", part)
print("AFTER: ", remove_stopwords(part))

Review index: 0
BEFORE: ['from', 'the', 'word', 'go', '.', 'trust', 'me', ',', 'this', 'is', 'not', 'a', 'show', 'for', 'the', 'faint', 'hearted', 'or', 'timid', '.']
AFTER:  ['word', 'go', '.', 'trust', ',', 'not', 'show', 'faint', 'hearted', 'timid', '.']


### 7.6 Statistics: how many tokens were removed?

In [64]:
tokens_before = df_processed["tokens"].apply(len).sum()
tokens_after = df_processed["tokens_no_stopwords"].apply(len).sum()

print("Total tokens before stopword removal:", tokens_before)
print("Total tokens after stopword removal: ", tokens_after)
print("Tokens removed:                      ", tokens_before - tokens_after,
      f"({(tokens_before - tokens_after) / tokens_before * 100:.1f}% of all tokens)")

removed_counts = Counter()
for tokens in df_processed["tokens"]:
    for token in tokens:
        if token in stop_words_to_remove:
            removed_counts[token] += 1

print("\nMost common removed stopwords:")
for word, count in removed_counts.most_common(15):
    print(f"  {word:8} -> {count} times")

Total tokens before stopword removal: 13191275
Total tokens after stopword removal:  7765868
Tokens removed:                       5425407 (41.1% of all tokens)

Most common removed stopwords:
  the      -> 663951 times
  and      -> 322677 times
  a        -> 321209 times
  of       -> 288587 times
  to       -> 267100 times
  is       -> 217184 times
  it       -> 189528 times
  in       -> 185335 times
  i        -> 173613 times
  this     -> 150414 times
  that     -> 143614 times
  was      -> 99998 times
  as       -> 91475 times
  with     -> 87171 times
  for      -> 86984 times


### 7.7 Check the results

In [65]:
stopwords_left = df_processed["tokens_no_stopwords"].apply(
    lambda tokens: sum(1 for token in tokens if token in stop_words_to_remove)
).sum()
print("Stopwords still left:", stopwords_left)

print("\nNegation words (before -> after):")
for word in ["not", "no", "never", "n't", "nothing"]:
    before = df_processed["tokens"].apply(lambda tokens: tokens.count(word)).sum()
    after = df_processed["tokens_no_stopwords"].apply(lambda tokens: tokens.count(word)).sum()
    print(f"  {word:8} {before} -> {after}")

print("\nReviews with no tokens left:", (df_processed["tokens_no_stopwords"].apply(len) == 0).sum())
print("Tokens in Step 6 column (unchanged):", df_processed["tokens"].apply(len).sum())

print("\nOriginal review column unchanged:", df["review"].equals(df_processed["review"]))
print("df_processed columns:", df_processed.columns.tolist())

Stopwords still left: 0

Negation words (before -> after):
  not      62414 -> 62414
  no       24629 -> 24629
  never    12888 -> 12888
  n't      66709 -> 66709
  nothing  8324 -> 8324

Reviews with no tokens left: 0
Tokens in Step 6 column (unchanged): 13191275

Original review column unchanged: True
df_processed columns: ['review', 'sentiment', 'normalized_review', 'cleaned_review', 'regex_review', 'lowercase_review', 'tokens', 'tokens_no_stopwords']


## Step 8: Stemming

### 8.1 Create the stemmer

In [66]:
from nltk.stem import PorterStemmer

porter_stemmer = PorterStemmer()

### 8.2 Demonstration on many words

In [67]:
demo_words = [
    "acting", "acted", "acts", "actor",
    "loved", "loving", "loves", "lovely",
    "watching", "watched", "watches",
    "movies", "movie", "films",
    "boring", "bored", "bores",
    "happiness", "happily", "studies", "studying",
    "entertainment", "entertaining", "wonderful", "better",
]

for word in demo_words:
    print(f"{word:15} -> {porter_stemmer.stem(word)}")

acting          -> act
acted           -> act
acts            -> act
actor           -> actor
loved           -> love
loving          -> love
loves           -> love
lovely          -> love
watching        -> watch
watched         -> watch
watches         -> watch
movies          -> movi
movie           -> movi
films           -> film
boring          -> bore
bored           -> bore
bores           -> bore
happiness       -> happi
happily         -> happili
studies         -> studi
studying        -> studi
entertainment   -> entertain
entertaining    -> entertain
wonderful       -> wonder
better          -> better


### 8.3 Apply stemming to all reviews

In [68]:
unique_tokens_no_stopwords = set()
for tokens in df_processed["tokens_no_stopwords"]:
    unique_tokens_no_stopwords.update(tokens)

stem_lookup = {word: porter_stemmer.stem(word) for word in unique_tokens_no_stopwords}

print("Unique tokens to stem:", len(stem_lookup))


def stem_tokens(tokens):
    return [stem_lookup[token] for token in tokens]


df_processed["tokens_stemmed"] = df_processed["tokens_no_stopwords"].apply(stem_tokens)

df_processed[["tokens_no_stopwords", "tokens_stemmed"]].head()

Unique tokens to stem: 152807


,tokens_no_stopwords,tokens_stemmed
0,"[one, reviewers, mentioned, watching, 1, oz, e...","[one, review, mention, watch, 1, oz, episod, '..."
1,"[wonderful, little, production, ., filming, te...","[wonder, littl, product, ., film, techniqu, un..."
2,"[thought, wonderful, way, spend, time, hot, su...","[thought, wonder, way, spend, time, hot, summe..."
3,"[basically, 's, family, little, boy, (, jake, ...","[basic, 's, famili, littl, boy, (, jake, ), th..."
4,"[petter, mattei, 's, ``, love, time, money, ''...","[petter, mattei, 's, ``, love, time, money, ''..."


### 8.4 Before-and-after examples

In [69]:
def show_stemming(index, n_tokens=25):
    before = df_processed.loc[index, "tokens_no_stopwords"][:n_tokens]
    after = df_processed.loc[index, "tokens_stemmed"][:n_tokens]
    print("Review index:", index)
    print("BEFORE (stopwords removed):", before)
    print()
    print("AFTER (stemmed):           ", after)
    print()
    print("Words that changed:")
    for word, stem in zip(before, after):
        if word != stem:
            print(f"  {word:15} -> {stem}")
    print("-" * 80)


show_stemming(0)
show_stemming(1)

Review index: 0
BEFORE (stopwords removed): ['one', 'reviewers', 'mentioned', 'watching', '1', 'oz', 'episode', "'ll", 'hooked', '.', 'right', ',', 'exactly', 'happened', '.', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scenes', 'violence', ',', 'set']

AFTER (stemmed):            ['one', 'review', 'mention', 'watch', '1', 'oz', 'episod', "'ll", 'hook', '.', 'right', ',', 'exactli', 'happen', '.', 'first', 'thing', 'struck', 'oz', 'brutal', 'unflinch', 'scene', 'violenc', ',', 'set']

Words that changed:
  reviewers       -> review
  mentioned       -> mention
  watching        -> watch
  episode         -> episod
  hooked          -> hook
  exactly         -> exactli
  happened        -> happen
  brutality       -> brutal
  unflinching     -> unflinch
  scenes          -> scene
  violence        -> violenc
--------------------------------------------------------------------------------
Review index: 1
BEFORE (stopwords removed): ['wonderful', 'little', 'production',

### 8.5 Many words become one stem

In [70]:
words_per_stem = {}
for word, stem in stem_lookup.items():
    words_per_stem.setdefault(stem, set()).add(word)

for stem in ["act", "love", "bore", "watch", "entertain"]:
    words = sorted(words_per_stem.get(stem, []))
    print(f"{stem:10} <- {words}")

act        <- ['act', 'acte', 'acted', 'acting', 'actings', 'acts']
love       <- ['love', 'loved', 'lovelies', 'lovely', 'loves', 'loving']
bore       <- ['bore', 'bored', 'bores', 'boring']
watch      <- ['watch', 'watched', 'watches', 'watchful', 'watchfulness', 'watching', 'watchings']
entertain  <- ['entertain', 'entertained', 'entertainement', 'entertainent', 'entertainer', 'entertainers', 'entertaining', 'entertainment', 'entertainments', 'entertains']


### 8.6 Statistics: how much smaller is the vocabulary?

In [71]:
unique_tokens_stemmed = set(stem_lookup.values())

print("Unique tokens before stemming:", len(unique_tokens_no_stopwords))
print("Unique tokens after stemming: ", len(unique_tokens_stemmed))
print("Vocabulary reduced by:        ", len(unique_tokens_no_stopwords) - len(unique_tokens_stemmed),
      f"({(1 - len(unique_tokens_stemmed) / len(unique_tokens_no_stopwords)) * 100:.1f}%)")

Unique tokens before stemming: 152807
Unique tokens after stemming:  120325
Vocabulary reduced by:         32482 (21.3%)


### 8.7 Check the results

In [72]:
length_mismatch = (
    df_processed["tokens_no_stopwords"].apply(len) != df_processed["tokens_stemmed"].apply(len)
).sum()
print("Reviews where the number of tokens changed:", length_mismatch)

print("\nNegation words after stemming:")
for word in ["not", "no", "never", "n't"]:
    print(f"  {word:6} -> {porter_stemmer.stem(word)}")

print("\nTokens in Step 7 column (unchanged):", df_processed["tokens_no_stopwords"].apply(len).sum())
print("Tokens in Step 8 column:            ", df_processed["tokens_stemmed"].apply(len).sum())

print("\nOriginal review column unchanged:", df["review"].equals(df_processed["review"]))
print("df_processed columns:", df_processed.columns.tolist())

Reviews where the number of tokens changed: 0

Negation words after stemming:
  not    -> not
  no     -> no
  never  -> never
  n't    -> n't

Tokens in Step 7 column (unchanged): 7765868
Tokens in Step 8 column:             7765868

Original review column unchanged: True
df_processed columns: ['review', 'sentiment', 'normalized_review', 'cleaned_review', 'regex_review', 'lowercase_review', 'tokens', 'tokens_no_stopwords', 'tokens_stemmed']
